# CODI Training on GPU - Simplified Version

**No CODI repo cloning needed!** Everything is bundled.

**GPU**: Free T4 → Runtime → Change runtime type → T4 GPU

**Time**: ~2 hours for 3 epochs on 6,000 examples

In [1]:
# Verify GPU is active
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

✅ GPU: Tesla T4
✅ Memory: 15.6 GB


In [2]:
# Clone your repo (includes codi_bundle with everything needed)
!git clone https://github.com/nabilanewaz/TokenSkip.git
%cd TokenSkip
!ls -lh codi_bundle/

Cloning into 'TokenSkip'...
remote: Enumerating objects: 322, done.
remote: Counting objects: 100% (204/204), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 322 (delta 59), reused 172 (delta 38), pack-reused 118 (from 1)
Receiving objects: 100% (322/322), 12.44 MiB | 8.62 MiB/s, done.
Resolving deltas: 100% (104/104), done.
/content/TokenSkip
total 32K
-rw-r--r-- 1 root root  765 Feb 25 15:23 README.md
drwxr-xr-x 2 root root 4.0K Feb 25 15:23 src
-rw-r--r-- 1 root root  21K Feb 25 15:23 train.py


In [3]:
# Install dependencies
!pip install peft==0.15.2 datasets==3.6.0 transformers==4.52.4 accelerate==1.7.0 safetensors -q
print("✅ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 136.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.3 MB/s eta 0:00:00
✅ Dependencies installed


In [4]:
# Verify training data
import json
train_file = 'datasets/gsm8k_split/llm_train.jsonl'
with open(train_file) as f:
    train_count = sum(1 for _ in f)
print(f"✅ Training data: {train_count} examples")

# Show first example
with open(train_file) as f:
    example = json.loads(f.readline())
print(f"\nExample: {example['question'][:100]}...")
print(f"Has CoT: {'cot' in example}")

✅ Training data: 6000 examples

Example: In a conference room, 40 chairs with a capacity of 2 people each were arranged in rows in preparatio...
Has CoT: True


In [5]:
# Download CODI checkpoint (GPT-2 + latent projection)
from huggingface_hub import snapshot_download
import os

ckpt_dir = snapshot_download(
    repo_id="zen-E/CODI-gpt2",
    ignore_patterns=["*.msgpack", "*.h5"]
)
print(f"✅ Checkpoint: {ckpt_dir}")

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/218 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/406M [00:00<?, ?B/s]

✅ Checkpoint: /root/.cache/huggingface/hub/models--zen-E--CODI-gpt2/snapshots/fd641b3d3edc59e4f534b55588e906588c9e36bb


In [6]:
# Prepare training data in CODI format
%cd codi_bundle

# Create datasets folder
!mkdir -p datasets/gsm8k
!cp ../datasets/gsm8k_split/llm_train.jsonl datasets/gsm8k/train.jsonl
!cp ../datasets/gsm8k_split/validation.jsonl datasets/gsm8k/val.jsonl
print("✅ Data prepared")

/content/TokenSkip/codi_bundle
✅ Data prepared


In [7]:
# Start training (adjust batch_size based on GPU memory)
# T4: batch_size=4, A100: batch_size=8+
!python train.py \
  --model_name_or_path gpt2 \
  --seed 42 \
  --model_max_length 512 \
  --lora_r 128 \
  --lora_alpha 32 \
  --lora_init \
  --num_latent 6 \
  --use_prj True \
  --prj_dim 768 \
  --inf_latent_iterations 6 \
  --remove_eos True \
  --use_lora True \
  --per_device_train_batch_size 4 \
  --data_name custom_local:datasets/gsm8k/train.jsonl \
  --output_dir ../outputs/codi_trained \
  --num_train_epochs 3 \
  --learning_rate 0.0002

Streaming output truncated to the last 5000 lines.
 45% 2020/4500 [18:25<22:02,  1.88it/s]latent5: distill_loss=0.8134765625
loss=1.6337890625, ce_loss=0.8203125, distill_loss=0.8134765625, ce_loss_total=0.8203125, distill_loss_total=0.8134765625, ref_ce_loss=1.50390625
 45% 2021/4500 [18:26<22:13,  1.86it/s]latent5: distill_loss=0.65380859375
loss=1.521484375, ce_loss=0.8671875, distill_loss=0.65380859375, ce_loss_total=0.8671875, distill_loss_total=0.65380859375, ref_ce_loss=1.5302734375
 45% 2022/4500 [18:26<24:25,  1.69it/s]latent5: distill_loss=0.6435546875
loss=1.3662109375, ce_loss=0.72265625, distill_loss=0.6435546875, ce_loss_total=0.72265625, distill_loss_total=0.6435546875, ref_ce_loss=1.1142578125
 45% 2023/4500 [18:27<25:04,  1.65it/s]latent5: distill_loss=0.71533203125
loss=1.51953125, ce_loss=0.80419921875, distill_loss=0.71533203125, ce_loss_total=0.80419921875, distill_loss_total=0.71533203125, ref_ce_loss=1.2529296875
 45% 2024/4500 [18:28<25:21,  1.63it/s]latent5: di

In [9]:
# Monitor progress (run this cell repeatedly)
!tail -n 30 ../outputs/codi_trained/*/logs.txt 2>/dev/null || echo "Check for log files in outputs/"

Check for log files in outputs/


In [10]:
# After training: check what was saved
%cd ..
!ls -lh outputs/codi_trained/

/content/TokenSkip
total 8.0K
drwxr-xr-x 3 root root 4.0K Feb 25 15:25 default
drwxr-xr-x 3 root root 4.0K Feb 25 15:25 runs


In [11]:
# Download checkpoint
!zip -r codi_checkpoint.zip outputs/codi_trained/
from google.colab import files
files.download('codi_checkpoint.zip')
print("✅ Checkpoint download started!")

  adding: outputs/codi_trained/ (stored 0%)
  adding: outputs/codi_trained/default/ (stored 0%)
  adding: outputs/codi_trained/default/gpt2/ (stored 0%)
  adding: outputs/codi_trained/default/gpt2/ep_3/ (stored 0%)
  adding: outputs/codi_trained/default/gpt2/ep_3/lr_0.0002/ (stored 0%)
  adding: outputs/codi_trained/default/gpt2/ep_3/lr_0.0002/seed_42/ (stored 0%)
  adding: outputs/codi_trained/default/gpt2/ep_3/lr_0.0002/seed_42/checkpoint-2000/ (stored 0%)
  adding: outputs/codi_trained/default/gpt2/ep_3/lr_0.0002/seed_42/checkpoint-2000/merges.txt (deflated 53%)
  adding: outputs/codi_trained/default/gpt2/ep_3/lr_0.0002/seed_42/checkpoint-2000/training_args.bin (deflated 53%)
  adding: outputs/codi_trained/default/gpt2/ep_3/lr_0.0002/seed_42/checkpoint-2000/scheduler.pt (deflated 61%)
  adding: outputs/codi_trained/default/gpt2/ep_3/lr_0.0002/seed_42/checkpoint-2000/trainer_state.json (deflated 78%)
  adding: outputs/codi_trained/default/gpt2/ep_3/lr_0.0002/seed_42/checkpoint-2000/r

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Checkpoint download started!


## Next: Phase 2 - Truth Vector Extraction

After downloading the checkpoint:

```powershell
# Extract locally
Expand-Archive codi_checkpoint.zip

# Phase 2: Extract truth vector from 500 steer examples
python extract_truth_vector.py --steer-data datasets/gsm8k_split/steer_train.jsonl
```

In [23]:
# Fix checkpoint path resolution
import pathlib

extract_tv_path = pathlib.Path('/content/TokenSkip/extract_truth_vector.py')
code = extract_tv_path.read_text()

# Patch get_checkpoint to use absolute paths
code = code.replace(
    'p = pathlib.Path(override)',
    'p = pathlib.Path(override).resolve()  # Convert to absolute path'
).replace(
    'p = pathlib.Path(f.read_text().strip())',
    'p = pathlib.Path(f.read_text().strip()).resolve()'
)

extract_tv_path.write_text(code)
print("✓ Patched extract_truth_vector.py")

✓ Patched extract_truth_vector.py


In [25]:
# Apply all path resolution fixes
import pathlib

extract_tv_path = pathlib.Path('/content/TokenSkip/extract_truth_vector.py')
code = extract_tv_path.read_text()

# Patch 1: get_checkpoint - use absolute checkpoint path
code = code.replace(
    'p = pathlib.Path(override)',
    'p = pathlib.Path(override).resolve()  # Convert to absolute path'
).replace(
    'p = pathlib.Path(f.read_text().strip())',
    'p = pathlib.Path(f.read_text().strip()).resolve()'
)

# Patch 2: run_dump - use absolute dump path
code = code.replace(
    'env["TV_DUMP_PATH"] = str(dump_path)',
    'env["TV_DUMP_PATH"] = str(dump_path.resolve())  # Use absolute path'
)

# Patch 3: main - resolve all directory paths
code = code.replace(
    'steer_data = pathlib.Path(args.steer_data)',
    'steer_data = pathlib.Path(args.steer_data).resolve()  # Convert to absolute path'
).replace(
    'out_dir    = pathlib.Path(args.out_dir)',
    'out_dir    = pathlib.Path(args.out_dir).resolve()  # Convert to absolute path'
)

extract_tv_path.write_text(code)
print("✓ Applied all path resolution patches to extract_truth_vector.py")

# Now run Phase 2
!python extract_truth_vector.py \
  --ckpt-dir outputs/codi_trained/default/gpt2/ep_3/lr_0.0002/seed_42/checkpoint-4500 \
  --steer-data datasets/gsm8k_split/steer_train.jsonl \
  --out-dir outputs/truth_vector \
  --batch-size 4

Streaming output truncated to the last 5000 lines.
Question 487 Ends
Prediction=2.0; Groundtruth=4.0

Question 488 Starts...
Q: Jana has 27 puppies. Two thirds of Jana's puppies are Pomeranians. One third of the Pomeranians are girls. How many girl Pomeranians does Jana have?
The answer is: 2
Question 488 Ends
Prediction=2.0; Groundtruth=6.0

Question 489 Starts...
Q: The highest temperature ever recorded in Southlandia is -48 degrees Fahrenheit. The highest temperature ever recorded in Northlandia is 21 degrees Fahrenheit. The highest temperature recorded in Midlandia is -3 degrees Fahrenheit. What is the average highest temperature of these 3 countries?
The answer is: 4
Question 489 Ends
Prediction=4.0; Groundtruth=-10.0

Question 490 Starts...
Q: An 8-year old child wants to buy a toy car which costs $12. He already has $4 savings. How many days will it take him to save the remaining amount of money if he promises to save $2 daily from his allowance?
The answer is: 4
Question 490 En